# Week 6/8 — AS/GLFT Theory Recovery

Tests the core research thesis: does a CVaR distributional RL agent rediscover
the GLFT closed-form policy in the low-volatility Poisson regime?

**Method:** Regress agent bid_offset vs inventory on the GLFT theoretical
skew curve. Report R² against GLFT (primary) and AS (upper bound).

**Target:** R² > 0.6 against GLFT for the best-performing agent.

**Inputs:** Trained checkpoints in `checkpoints/`  
**Outputs:** `experiments/w06_as_recovery/`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from evaluation.as_recovery import (
    run_recovery_analysis,
    run_recovery_all_agents,
    recovery_summary_df,
    glft_skew_curve,
    as_skew_curve,
    empirical_skew_curve,
    collect_skew_data,
)
from evaluation.visualize import Visualizer, quote_skew_curve
from training.evaluate import load_agent
from envs.lob_env import LOBMarketMakingEnv

CKPT_ROOT = Path('../checkpoints')
EXP_DIR   = Path('../experiments/w06_as_recovery')
EXP_DIR.mkdir(parents=True, exist_ok=True)

AGENTS  = ['sarsa', 'dqn', 'ppo', 'qrdqn', 'iqn']
REGIME  = 'low_vol'
ENCODER = 'handcrafted'
REWARD  = 'asymmetric'
SEED    = 42

# Baseline calibration params (match Week 3 baseline_quote_skew figure)
GAMMA     = 1.0
KAPPA     = 19.5
SIGMA     = 0.245
XI        = 0.0
A         = 1.0
Q_MAX     = 10
T         = 390.0
TICK_SIZE = 0.01
TAU_HAT   = 0.5   # mid-episode evaluation point

def run_tag(agent):
    return f'{agent}_{ENCODER}_{REWARD}_{REGIME}_seed{SEED}'

viz = Visualizer(out_root=EXP_DIR)
print('Setup complete')

## 1 — Theoretical reference curves

In [ ]:
# Compute GLFT and AS theoretical curves at each inventory level
inv_range   = np.arange(-Q_MAX + 2, Q_MAX - 1)   # exclude boundary levels

glft_theory = glft_skew_curve(
    inv_range, GAMMA, KAPPA, SIGMA, XI, A, Q_MAX, T, TICK_SIZE, TAU_HAT
)
as_theory   = as_skew_curve(
    inv_range, GAMMA, KAPPA, SIGMA, TICK_SIZE, TAU_HAT
)

print('Theoretical skew curves (ticks below mid):')
print(f'{"q":>4}  {"GLFT":>8}  {"AS":>8}')
print('─' * 24)
for q, g, a in zip(inv_range, glft_theory, as_theory):
    if not np.isnan(g):
        print(f'{q:>4}  {g:>8.2f}  {a:>8.2f}')

## 2 — Load agents and collect rollout data

In [ ]:
env = LOBMarketMakingEnv(
    reward_type='asymmetric', episode_len=390,
    Q_max=Q_MAX, tick_size=TICK_SIZE, seed=2000, use_abides=False,
)

loaded_agents = {}
for agent_name in AGENTS:
    ext  = '.npz' if agent_name == 'sarsa' else '.pt'
    cdir = CKPT_ROOT / run_tag(agent_name)
    cks  = sorted([c for c in cdir.glob(f'*{ext}')
                   if '_meta' not in c.name]) if cdir.exists() else []
    if not cks:
        print(f'  [skip] {agent_name} — no checkpoint')
        continue
    try:
        ag, enc_type = load_agent(str(cks[-1]), agent_name, ENCODER)
        loaded_agents[agent_name] = (ag, enc_type)
        print(f'  Loaded {agent_name}')
    except Exception as e:
        print(f'  {agent_name}: {e}')

print(f'Loaded {len(loaded_agents)} agents')

## 3 — Run recovery analysis for all agents

In [ ]:
N_EVAL_EPS = 20   # increase to 50 for publication
all_results = {}

for agent_name, (agent, enc_type) in loaded_agents.items():
    print(f'\n{"─"*50}')
    print(f'Recovery analysis: {agent_name}')
    result = run_recovery_analysis(
        agent      = agent,
        env        = env,
        enc_type   = enc_type,
        n_episodes = N_EVAL_EPS,
        gamma      = GAMMA,
        kappa      = KAPPA,
        sigma      = SIGMA,
        xi         = XI,
        A          = A,
        Q_max      = Q_MAX,
        T          = T,
        tick_size  = TICK_SIZE,
        tau_hat    = TAU_HAT,
        seed       = 2000,
    )
    all_results[agent_name] = result

env.close()

## 4 — R² summary table

In [ ]:
df_summary = recovery_summary_df(all_results)
display(df_summary)
df_summary.to_csv(EXP_DIR / 'recovery_summary.csv', index=False)
print(f'\nSaved → {EXP_DIR}/recovery_summary.csv')
print()
print('Interpretation:')
print('  R² > 0.6 = strong recovery (agent rediscovered GLFT structure)')
print('  R² > 0.5 = moderate recovery')
print('  R² < 0.3 = no recovery')

## 5 — Figure: empirical vs theoretical skew per agent

In [ ]:
from evaluation.visualize import THEME, AGENT_COLORS, _dark_fig, _label, _legend

n_agents_with_data = sum(
    1 for r in all_results.values()
    if r.get('glft_fit') is not None
)

if n_agents_with_data == 0:
    print('No recovery data — run training first')
else:
    ncols = min(n_agents_with_data, 3)
    nrows = (n_agents_with_data + ncols - 1) // ncols
    fig, axes = _dark_fig(figsize=(5 * ncols, 4 * nrows),
                          nrows=nrows, ncols=ncols)
    if nrows * ncols == 1:
        axes = np.array([[axes]])

    ax_flat = axes.flat
    for agent_name, result in all_results.items():
        if result.get('glft_fit') is None:
            continue
        ax    = next(ax_flat)
        color = AGENT_COLORS.get(agent_name, '#ffffff')
        invs  = result['inv_levels']
        emp   = result['empirical']
        glft  = result['glft_theory']
        as_   = result['as_theory']

        ax.plot(invs, emp,  color=color, lw=2, label='Agent', marker='o', ms=4)
        ax.plot(invs, glft, color='#06b6d4', lw=1.5, ls='--', label='GLFT')
        ax.plot(invs, as_,  color='#f59e0b', lw=1.5, ls=':',  label='AS')

        r2_g = result['glft_fit']['r2']
        r2_a = result['as_fit']['r2']
        ax.set_title(f'{agent_name}  R²(GLFT)={r2_g:.3f}  R²(AS)={r2_a:.3f}',
                     color=THEME['text'], fontsize=10, pad=6)
        _legend(ax, fontsize=8)
        _label(ax, xlabel='Inventory q', ylabel='Bid offset (ticks)')

    # Hide unused axes
    for ax in ax_flat:
        ax.set_visible(False)

    fig.suptitle('AS/GLFT Theory Recovery — Empirical vs Theoretical Skew\n'
                 f'low_vol regime · {N_EVAL_EPS} eval episodes each',
                 color=THEME['text'], fontsize=12, y=1.01)
    fig.tight_layout()
    path = EXP_DIR / 'recovery_curves.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=THEME['bg'])
    plt.close(fig)
    print(f'Saved → {path}')

## 6 — Regression diagnostics for best agent

In [ ]:
if df_summary.empty:
    print('No data')
else:
    best = df_summary.iloc[0]
    name = best['agent']
    r    = all_results[name]

    print(f'Best agent: {name}')
    print(f'{'─'*45}')
    print(f'GLFT fit:')
    for k, v in r['glft_fit'].items():
        print(f'  {k:12s}: {v}')
    print()
    print(f'AS fit:')
    for k, v in r['as_fit'].items():
        print(f'  {k:12s}: {v}')
    print()
    print(r['summary'])

    # Slope interpretation
    slope = r['glft_fit']['slope']
    if abs(slope - 1.0) < 0.2:
        print('\nSlope ≈ 1.0 — agent matches GLFT magnitude')
    elif slope < 0.5:
        print('\nSlope < 0.5 — agent is under-skewing vs GLFT')
        print('Check: is reward eta actually penalising inventory?')
    elif slope > 1.5:
        print('\nSlope > 1.5 — agent is over-skewing vs GLFT')

## 7 — Flat skew debug (if needed)

In [ ]:
# Run this cell if any agent shows flat bid_offset vs q (no skew)
# Confirms whether reward eta is actually penalising inventory

DEBUG_AGENT = 'qrdqn'   # change to whichever agent is flat

if DEBUG_AGENT in all_results and all_results[DEBUG_AGENT].get('glft_fit'):
    r2 = all_results[DEBUG_AGENT]['glft_fit']['r2']
    if r2 < 0.1:
        print(f'{DEBUG_AGENT} shows near-zero R² — running reward debug')
        print()

        # Quick reward component check
        env_dbg = LOBMarketMakingEnv(
            reward_type='asymmetric', episode_len=20,
            Q_max=Q_MAX, tick_size=TICK_SIZE, seed=999, use_abides=False,
        )
        obs, info = env_dbg.reset()
        print('Step | inv_pnl | spread_pnl | penalty | reward')
        print('─' * 55)
        for step in range(5):
            action = env_dbg.action_space.sample()
            obs, reward, term, trunc, info = env_dbg.step(action)
            inv_pnl    = info.get('inv_pnl', 0.0)
            spread_pnl = info.get('spread_pnl', 0.0)
            penalty    = max(0.0, 0.5 * inv_pnl)   # eta=0.5
            print(f'{step:4d} | {inv_pnl:7.4f} | {spread_pnl:10.4f} '
                  f'| {penalty:7.4f} | {reward:6.4f}')
            if term or trunc:
                break
        env_dbg.close()

        print()
        print('If penalty is always 0: inv_pnl is always <= 0')
        print('or eta is not being applied. Check lob_env.py reward calculation.')
    else:
        print(f'{DEBUG_AGENT} R²={r2:.3f} — skew is present, no debug needed')
else:
    print(f'{DEBUG_AGENT} results not available')

## 8 — Save results

In [ ]:
import json

serialisable = {}
for agent_name, r in all_results.items():
    if r.get('glft_fit') is None:
        continue
    serialisable[agent_name] = {
        'inv_levels':  r['inv_levels'].tolist(),
        'empirical':   r['empirical'].tolist(),
        'glft_theory': r['glft_theory'].tolist() if r['glft_theory'] is not None else None,
        'as_theory':   r['as_theory'].tolist()   if r['as_theory']   is not None else None,
        'glft_fit':    r['glft_fit'],
        'as_fit':      r['as_fit'],
        'summary':     r['summary'],
    }

out_path = EXP_DIR / 'recovery_results.json'
with open(out_path, 'w') as f:
    json.dump(serialisable, f, indent=2)
print(f'Saved → {out_path}')
print()
print('Files written:')
for p in sorted(EXP_DIR.glob('*')):
    print(f'  {p.name}')